# **Data Cleaning**

### **Objective**
The goal here is to fix structural and
quality issues in the data — **not** to impute missing values or scale/encode
features. Imputation, encoding, and scaling are intentionally happens *after* the train/test split,
so that no information from the test set leaks into the training process.


**1-Import Libraries**

In [49]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import f_classif, chi2, mutual_info_classif


**2-Load Dataset**

In [50]:
df = pd.read_csv("Loan_Default.csv")
print(df.shape)
df.head()


(148670, 34)


,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


**3-Remove Duplicate Rows**

In [51]:
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
print(f"Removed {before - after} duplicate rows")


Removed 0 duplicate rows


**4- Detect Problems**

In [52]:
numerical_features = df.select_dtypes(include=np.number).columns.tolist()
numerical_features.remove("Status")
numerical_features

['ID',
 'year',
 'loan_amount',
 'rate_of_interest',
 'Interest_rate_spread',
 'Upfront_charges',
 'term',
 'property_value',
 'income',
 'Credit_Score',
 'LTV',
 'dtir1']

In [53]:
for cols in numerical_features:
  print(cols)
  print(df[cols].unique().tolist())
  print("-"*50)

ID
[24890, 24891, 24892, 24893, 24894, 24895, 24896, 24897, 24898, 24899, 24900, 24901, 24902, 24903, 24904, 24905, 24906, 24907, 24908, 24909, 24910, 24911, 24912, 24913, 24914, 24915, 24916, 24917, 24918, 24919, 24920, 24921, 24922, 24923, 24924, 24925, 24926, 24927, 24928, 24929, 24930, 24931, 24932, 24933, 24934, 24935, 24936, 24937, 24938, 24939, 24940, 24941, 24942, 24943, 24944, 24945, 24946, 24947, 24948, 24949, 24950, 24951, 24952, 24953, 24954, 24955, 24956, 24957, 24958, 24959, 24960, 24961, 24962, 24963, 24964, 24965, 24966, 24967, 24968, 24969, 24970, 24971, 24972, 24973, 24974, 24975, 24976, 24977, 24978, 24979, 24980, 24981, 24982, 24983, 24984, 24985, 24986, 24987, 24988, 24989, 24990, 24991, 24992, 24993, 24994, 24995, 24996, 24997, 24998, 24999, 25000, 25001, 25002, 25003, 25004, 25005, 25006, 25007, 25008, 25009, 25010, 25011, 25012, 25013, 25014, 25015, 25016, 25017, 25018, 25019, 25020, 25021, 25022, 25023, 25024, 25025, 25026, 25027, 25028, 25029, 25030, 25031, 25

**5-Fix Inconsistent Categorical Values**


In [54]:
categorical_features = df.select_dtypes(include="object").columns.tolist()
categorical_features

['loan_limit',
 'Gender',
 'approv_in_adv',
 'loan_type',
 'loan_purpose',
 'Credit_Worthiness',
 'open_credit',
 'business_or_commercial',
 'Neg_ammortization',
 'interest_only',
 'lump_sum_payment',
 'construction_type',
 'occupancy_type',
 'Secured_by',
 'total_units',
 'credit_type',
 'co-applicant_credit_type',
 'age',
 'submission_of_application',
 'Region',
 'Security_Type']

In [55]:
for col in categorical_features:
    print(col)
    print(df[col].unique().tolist())
    print("-" * 50)

loan_limit
['cf', nan, 'ncf']
--------------------------------------------------
Gender
['Sex Not Available', 'Male', 'Joint', 'Female']
--------------------------------------------------
approv_in_adv
['nopre', 'pre', nan]
--------------------------------------------------
loan_type
['type1', 'type2', 'type3']
--------------------------------------------------
loan_purpose
['p1', 'p4', 'p3', 'p2', nan]
--------------------------------------------------
Credit_Worthiness
['l1', 'l2']
--------------------------------------------------
open_credit
['nopc', 'opc']
--------------------------------------------------
business_or_commercial
['nob/c', 'b/c']
--------------------------------------------------
Neg_ammortization
['not_neg', 'neg_amm', nan]
--------------------------------------------------
interest_only
['not_int', 'int_only']
--------------------------------------------------
lump_sum_payment
['not_lpsm', 'lpsm']
--------------------------------------------------
construction_ty

In [56]:
df["Gender"] = df["Gender"].replace("Sex Not Available", np.nan)
df["Gender"].value_counts(dropna=False)


,count
Gender,
Male,42346
Joint,41399
NaN,37659
Female,27266


In [57]:
df["Security_Type"] = df["Security_Type"].replace("Indriect", "Indirect")
df["Security_Type"].value_counts()


,count
Security_Type,
direct,148637
Indirect,33


In [58]:
df["Region"] = df["Region"].str.lower()
df["Region"].value_counts()


,count
Region,
north,74722
south,64016
central,8697
north-east,1235


In [59]:
for col in categorical_features:
    df[col] = df[col].str.strip()


In [60]:
for col in categorical_features:
    print(col)
    print(df[col].unique().tolist())
    print("-" * 50)

loan_limit
['cf', nan, 'ncf']
--------------------------------------------------
Gender
[nan, 'Male', 'Joint', 'Female']
--------------------------------------------------
approv_in_adv
['nopre', 'pre', nan]
--------------------------------------------------
loan_type
['type1', 'type2', 'type3']
--------------------------------------------------
loan_purpose
['p1', 'p4', 'p3', 'p2', nan]
--------------------------------------------------
Credit_Worthiness
['l1', 'l2']
--------------------------------------------------
open_credit
['nopc', 'opc']
--------------------------------------------------
business_or_commercial
['nob/c', 'b/c']
--------------------------------------------------
Neg_ammortization
['not_neg', 'neg_amm', nan]
--------------------------------------------------
interest_only
['not_int', 'int_only']
--------------------------------------------------
lump_sum_payment
['not_lpsm', 'lpsm']
--------------------------------------------------
construction_type
['sb', 'mh']


**6-Logical / Range Checks**


In [61]:
numeric_cols_to_check = ["loan_amount", "rate_of_interest", "income","property_value", "LTV", "dtir1", "term"]
for col in numeric_cols_to_check:
    n_negative = (df[col] < 0).sum()
    print(f"{col}: {n_negative} negative values")


loan_amount: 0 negative values
rate_of_interest: 0 negative values
income: 0 negative values
property_value: 0 negative values
LTV: 0 negative values
dtir1: 0 negative values
term: 0 negative values


In [62]:
df["LTV"].describe()


,LTV
count,133572.000000
mean,72.746457
std,39.967603
min,0.967478
25%,60.474860
50%,75.135870
75%,86.184211
max,7831.250000


**7-Data Type Check**

Confirm that numerical columns are stored as numeric dtypes and categorical
columns as object/category.

In [63]:
df.dtypes


,0
ID,int64
year,int64
loan_limit,object
Gender,object
approv_in_adv,object
loan_type,object
loan_purpose,object
Credit_Worthiness,object
open_credit,object
business_or_commercial,object


**8-Note on Missing Values**

`rate_of_interest`, `Interest_rate_spread`, `Upfront_charges`, `dtir1`,
`property_value`, `LTV`, `income`, `loan_limit`, `approv_in_adv`,
`loan_purpose`, `age`, `submission_of_application`, and now `Gender` still
contain missing values.

Imputation will be fitted on the
training set only , after the
train/test split, to avoid data leakage .

In [64]:
df.isnull().sum().sort_values(ascending=False)


,0
Upfront_charges,39642
Gender,37659
Interest_rate_spread,36639
rate_of_interest,36439
dtir1,24121
property_value,15098
LTV,15098
income,9150
loan_limit,3344
approv_in_adv,908


**9-Save Cleaned Dataset**

In [65]:
df.to_csv("loan_data_cleaned.csv", index=False)
print("Saved cleaned dataset:", df.shape)


Saved cleaned dataset: (148670, 34)


# **Feature Selection & Row Sampling**

### **Objective**


1. **Reduces the dataset from 148,670 rows to 50,000 rows**, using
   **stratified sampling** on the target (`Status`) so the 75.36% / 24.64%
   class balance found in the EDA is preserved. A random (non-stratified)
   sample could accidentally shift that ratio and make the imbalance problem
   worse or better than it really is.
2. **Selects features** using a mix of domain knowledge (from Data
   Understanding / EDA)

**1- Clean Dataset**

In [66]:
df = pd.read_csv("loan_data_cleaned.csv")
print(df.shape)
df.head()


(148670, 34)


,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,NaN,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,north,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,north,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,north,direct,0,39.0


**2-Drop Irrelevant / Constant Features**


- `ID` is a unique identifier — no predictive value.
- `year` has a single constant value (2019) — no predictive value.

In [67]:
drop_irrelevant = ["ID", "year"]
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
print("Constant columns found:", constant_cols)

drop_irrelevant = list(set(drop_irrelevant + constant_cols))
df = df.drop(columns=[c for c in drop_irrelevant if c in df.columns])
print("Dropped:", drop_irrelevant)
print(df.shape)


Constant columns found: ['year']
Dropped: ['year', 'ID']
(148670, 32)


**3-Reduce Dataset Size to 50,000 Rows**

`train_test_split` is used here purely as a convenient stratified-sampling
tool (not as the real train/test split ). We keep the 50,000-row portion and discard the rest.

In [68]:
sample_size = 50000
df_sampled, _ = train_test_split(df,train_size=sample_size,stratify=df["Status"],random_state=42)
df_sampled = df_sampled.reset_index(drop=True)
print(df_sampled.shape)
df_sampled["Status"].value_counts(normalize=True) * 100


(50000, 32)


,proportion
Status,
0,75.356
1,24.644


**4-Save Reduced & Selected Dataset**

In [78]:
df_sampled.to_csv("loan_data_sampled.csv", index=False)

print("Saved sampled dataset:", df_sampled.shape)


Saved sampled dataset: (50000, 32)
